In [1]:
library(numbat)
library(dplyr)
library(Seurat)
library(data.table)
library(future)
plan("multicore", workers = 12)
options(future.globals.maxSize = 10000 * 1024^10,
        future.rng.onMisuse = 'ignore')
sessionInfo()

Loading required package: Matrix


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, saveRDS


Loading Seurat v5 beta version 
To maintain compatibility with previous workflows, new Seurat objects will use the previous object structure by default
To use new Seurat v5 assays: Please run: options(Seurat.object.assay.version = 'v5')


Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




R version 4.2.3 (2023-03-15)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Rocky Linux 8.7 (Green Obsidian)

Matrix products: default
BLAS/LAPACK: /gpfs/home3/cruiz2/miniconda3/envs/r_python/lib/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] future_1.32.0           data.table_1.14.8       Seurat_4.9.9.9040      
[4] SeuratObject_4.9.9.9081 sp_1.6-0                dplyr_1.1.1            
[7] numbat_1.2.3            Matrix_1.5-3           

loaded via a namespace (and not attached):
  [1] uuid_1.1-0             spam_

In [2]:
dmg <- readRDS('/projects/0/einf2548/cruiz/dmg/data/merged_dmg_atlas_qc_filtered.rds')
dmg

An object of class Seurat 
38576 features across 409561 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [4]:
dmg$numbat_id <- recode(dmg$predicted.annotation_level_3,
                         `AC-like` = 'Malignant',
                                       `MES-like` = "Malignant",    
                                       `OPC-like` = "Malignant", 
                                       `NPC-like` = "Malignant",    
                                       `Astrocyte` = "Glial_Neuronal", 
                                       # `Oligodendrocyte` = "Glial_Neuronal",    
                                       `OPC` = "Glial_Neuronal", 
                                       `Neuron` = "Glial_Neuronal",
                                        `Mono` = "Myeloid",
                                       `TAM-BDM` = "Myeloid",    
                                       `TAM-MG` = "Myeloid", 
                                       `DC` = "Myeloid",    
                                       `Mast` = "Myeloid",
                                       `CD4/CD8` = "Lymphoid",    
                                       `NK` = "Lymphoid", 
                                       `B cell` = "Lymphoid",    
                                       `Plasma B` = "Lymphoid",
                                       `Endothelial` = "Vascular",    
                                       `Mural cell` = "Vascular"
                         )
Idents(dmg) <- dmg$numbat_id
table(dmg@active.ident)


      Malignant  Glial_Neuronal        Vascular         Myeloid Oligodendrocyte 
         231886           78710            7188           60905           25955 
       Lymphoid 
           4917 

In [5]:
`%nin%` <- function(x, y) !(x %in% y)
dmg <- subset(dmg, Study %nin% c('Filbin2018', 'Liu2022'))
dmg
gc()

An object of class Seurat 
38576 features across 397794 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,8382601,447.7,16414148,876.7,16414148,876.7
Vcells,12653726388,96540.3,16213889217,123702.2,12897181282,98397.7


In [8]:
dmg <- subset(dmg, Batch_for_correction == '10Xv3_cell_rna')
dmg

An object of class Seurat 
38576 features across 57305 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [9]:
ref_internal = aggregate_counts(GetAssayData(subset(dmg, 
                                                   idents = c('Malignant', 'Glial_Neuronal'),
                                                   invert = TRUE),
                                             slot = 'counts'
                                            ), 
                               as.data.frame(dmg@active.ident) %>% 
                                `colnames<-`('group') %>% 
                                tibble::rownames_to_column('cell') %>%
                                filter(!group %in% c('Malignant', 'Glial_Neuronal'))
                               )
ref_internal

Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.”


cell_dict
       Vascular         Myeloid Oligodendrocyte        Lymphoid 
            494            9876             274             385 


,Vascular,Myeloid,Oligodendrocyte,Lymphoid
A1BG,2.026477e-05,3.359908e-06,2.672274e-05,1.608720e-05
A1CF,0.000000e+00,2.181759e-08,0.000000e+00,0.000000e+00
A2M,9.342580e-04,7.911493e-04,1.027798e-05,5.692393e-05
A2ML1,5.196095e-07,3.490814e-07,0.000000e+00,0.000000e+00
A3GALT2,1.039219e-06,1.614501e-06,2.055596e-06,0.000000e+00
A4GALT,2.182360e-05,6.545276e-08,0.000000e+00,0.000000e+00
A4GNT,0.000000e+00,1.527231e-07,0.000000e+00,0.000000e+00
AAAS,9.872581e-06,6.719816e-06,3.288953e-05,3.712430e-06
AACS,1.299024e-05,5.432579e-06,1.233357e-05,1.113729e-05
AADAC,0.000000e+00,6.545276e-08,0.000000e+00,0.000000e+00


### Process Jessa2022

In [10]:
jessa <- SplitObject(dmg, split.by = 'SampleID')
jessa
gc()

$`P-6117_S-8370_RNA_only`
An object of class Seurat 
38576 features across 12688 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

$`P-6240_S-8628_RNA_only`
An object of class Seurat 
38576 features across 6680 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

$`P-6328_S-8672_RNA_only`
An object of class Seurat 
38576 features across 13392 samples with

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,8138678,434.7,16414148,876.7,16414148,876.7
Vcells,7784315822,59389.7,16213889217,123702.2,14401538685,109875.1


In [11]:
table(dmg$SampleID)


P-6117_S-8370_RNA_only P-6240_S-8628_RNA_only P-6328_S-8672_RNA_only 
                 12688                   6680                  13392 
P-6337_S-8821_RNA_only P-6519_S-9084_RNA_only 
                 12318                  12227 

In [12]:
rm(dmg)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,8135632,434.5,16414148,876.7,16414148,876.7
Vcells,6971516305,53188.5,16213889217,123702.2,14401538685,109875.1


In [13]:
#check cell name nomenclature to add to numbat pipeline
head(colnames(jessa[[1]]))

[1] "rna_P-6117_S-8370_TGGCCAGCAGCGTAAG-1"
[2] "rna_P-6117_S-8370_CAAGAAACAACTGCTA-1"
[3] "rna_P-6117_S-8370_GCTGCAGCACCATCCT-1"
[4] "rna_P-6117_S-8370_GCGAGAACACTCGACG-1"
[5] "rna_P-6117_S-8370_GCAGCCACAGTGACAG-1"
[6] "rna_P-6117_S-8370_GACCAATTCCCTAATT-1"

In [14]:
length(jessa)

[1] 5

In [15]:
names(jessa) <- gsub('_RNA_only', '', names(jessa))
jessa

$`P-6117_S-8370`
An object of class Seurat 
38576 features across 12688 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

$`P-6240_S-8628`
An object of class Seurat 
38576 features across 6680 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

$`P-6328_S-8672`
An object of class Seurat 
38576 features across 13392 samples within 6 assays 
Active assay: 

In [16]:
for(i in 1:length(jessa)) {
    cat(' ############################################\n',
        '### Numbat for dataset number ', names(jessa[i]), '###\n',
        '############################################\n')
    
    old <- Sys.time() # get start time
    
    file_path <- paste0('/projects/0/einf2548/cruiz/dmg/public_data/Jessa2022/EGAD00001008351_sc_sn_RNA/numbat_pileup_output_cellbender/',
                        names(jessa[i]),'/',names(jessa[i]),'_allele_counts.tsv.gz')
    
    if (file.exists(file_path)) {  # check if file exists
        df_allele = fread(file_path)
        df_allele$cell <- paste0('rna_', names(jessa[i]),'_',df_allele$cell)
          
        # run
        out = run_numbat(
            as.matrix(GetAssayData(jessa[[i]], slot = 'counts', assay = 'RNA')), # gene x cell integer UMI count matrix 
            ref_internal, # reference expression profile, a gene x cell type normalized expression level matrix
            df_allele, # allele dataframe generated by pileup_and_phase script and MUST BE A DATATABLE!
            min_cells = 20,
            t = 1e-3,
            max_iter = 2,
            init_k = 3,
            ncores = 40,
            ncores_nni = 40,
            multi_allelic = TRUE,
            plot = TRUE,
            genome = "hg38",
            out_dir = paste0('/projects/0/einf2548/cruiz/dmg/notebooks/iCNV/numbat/dmg_atlas_segregated_ref/Jessa2022/sc/',names(jessa[i]))
        )
      
        cat(' ############################################\n',
        '### Done with dataset', names(jessa[i]), '###\n',
        '############################################\n')
        
        # print elapsed time
        new <- Sys.time() - old # calculate difference
        print(new) # print in nice format
    } else {
        cat(' ############################################\n',
        '### Skipping dataset', names(jessa[i]), 'because the file does not exist ###\n',
        '############################################\n')
    }
}


 ############################################
 ### Numbat for dataset number  P-6117_S-8370 ###
 ############################################


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.8 GiB”
Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 3
max_cost = 3806.4
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 0.5
skip_nj = FALSE
diploid_chroms = 
ncores = 40
ncores_nni = 40
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
12688 cells

Mem used: 58.7Gb

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.4 GiB”
Approximating initial clusters using smoothed expression ..

Mem used: 58.7Gb

number of genes left: 11591

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.1 GiB”
running hclust...

Iteration 1

Mem used: 23Gb

Running HMMs on 5 cell groups..

Expression noise level: high (1). Consider using a custom express

 ############################################
 ### Done with dataset P-6117_S-8370 ###
 ############################################
Time difference of 3.138038 hours
 ############################################
 ### Numbat for dataset number  P-6240_S-8628 ###
 ############################################


Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 3
max_cost = 2004
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 0.5
skip_nj = FALSE
diploid_chroms = 
ncores = 40
ncores_nni = 40
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
6680 cells

Mem used: 9.07Gb

Approximating initial clusters using smoothed expression ..

Mem used: 9.05Gb

number of genes left: 12048

running hclust...

Iteration 1

Mem used: 16.7Gb

Running HMMs on 5 cell groups..

Expression noise level: medium (0.9). 

Running HMMs on 3 cell groups..

Testing for multi-allelic CNVs ..

0 multi-allelic CNVs found: 

Evaluating CNV per cell ..

Mem used: 9.76Gb

All cells succeeded

Expanding allelic states..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

Building phyloge

 ############################################
 ### Done with dataset P-6240_S-8628 ###
 ############################################
Time difference of 1.688322 hours
 ############################################
 ### Numbat for dataset number  P-6328_S-8672 ###
 ############################################


...

 ############################################
 ### Done with dataset P-6328_S-8672 ###
 ############################################
Time difference of 9.51719 hours
 ############################################
 ### Numbat for dataset number  P-6337_S-8821 ###
 ############################################


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.8 GiB”
Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 3
max_cost = 3695.4
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 0.5
skip_nj = FALSE
diploid_chroms = 
ncores = 40
ncores_nni = 40
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
12318 cells

Mem used: 10.7Gb

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.3 GiB”
Approximating initial clusters using smoothed expression ..

Mem used: 10.7Gb

number of genes left: 12174

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.1 GiB”
running hclust...

Iteration 1

Mem used: 23.6Gb

Running HMMs on 5 cell groups..

Expression noise level: medium (0.8). 

Running HMMs on 3 cell 

 ############################################
 ### Done with dataset P-6337_S-8821 ###
 ############################################
Time difference of 2.572544 hours
 ############################################
 ### Numbat for dataset number  P-6519_S-9084 ###
 ############################################


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.8 GiB”
Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 3
max_cost = 3668.1
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 0.5
skip_nj = FALSE
diploid_chroms = 
ncores = 40
ncores_nni = 40
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
12227 cells

Mem used: 10.9Gb

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.3 GiB”
Approximating initial clusters using smoothed expression ..

Mem used: 10.9Gb

number of genes left: 11917

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.1 GiB”
running hclust...

Iteration 1

Mem used: 23.5Gb

Running HMMs on 5 cell groups..

Expression noise level: medium (0.69). 

Running HMMs on 3 cell

 ############################################
 ### Done with dataset P-6519_S-9084 ###
 ############################################
Time difference of 2.302198 hours
